###  1. Importing Libraries.

In [1]:
import pandas as pd  
pd.set_option("display.max_columns", None)  
import numpy as np  
from sklearn.model_selection import train_test_split  
from sklearn.preprocessing import MinMaxScaler  

import torch  
import torch.nn as nn  
from torch.utils.data import TensorDataset, DataLoader  

###  2. Reading and Inspecting the Dataset

In [2]:
dataframe = pd.read_csv("dataset_phishing.csv")  
print(dataframe.head(10))

                                                 url  length_url  \
0              http://www.crestonwood.com/router.php          37   
1  http://shadetreetechnology.com/V4/validation/a...          77   
2  https://support-appleld.com.secureupdate.duila...         126   
3                                 http://rgipt.ac.in          18   
4  http://www.iracing.com/tracks/gateway-motorspo...          55   
5                   http://appleid.apple.com-app.es/          32   
6                                http://www.mutuo.it          19   
7  http://www.shadetreetechnology.com/V4/validati...          81   
8         http://vamoaestudiarmedicina.blogspot.com/          42   
9  https://parade.com/425836/joshwigler/the-amazi...         104   

   length_hostname  ip  nb_dots  nb_hyphens  nb_at  nb_qm  nb_and  nb_or  \
0               19   0        3           0      0      0       0      0   
1               23   1        1           0      0      0       0      0   
2               50   1 

###  3. Preprocessing: Dropping the URL Column

In [3]:
dataframe.drop(["url"], axis=1, inplace=True)
print(dataframe.head())

   length_url  length_hostname  ip  nb_dots  nb_hyphens  nb_at  nb_qm  nb_and  \
0          37               19   0        3           0      0      0       0   
1          77               23   1        1           0      0      0       0   
2         126               50   1        4           1      0      1       2   
3          18               11   0        2           0      0      0       0   
4          55               15   0        2           2      0      0       0   

   nb_or  nb_eq  nb_underscore  nb_tilde  nb_percent  nb_slash  nb_star  \
0      0      0              0         0           0         3        0   
1      0      0              0         0           0         5        0   
2      0      3              2         0           0         5        0   
3      0      0              0         0           0         2        0   
4      0      0              0         0           0         5        0   

   nb_colon  nb_comma  nb_semicolumn  nb_dollar  nb_space  nb_

###  4. Converting Class Labels to Numbers

In [4]:
class_labels = dataframe['status'].unique().tolist()  
class_labels.sort()  
class_dict = {label: idx for idx, label in enumerate(class_labels)}  
dataframe['status'] = dataframe['status'].map(class_dict)

### 5. Separating Features and Labels

In [5]:
X = dataframe.iloc[:, :-1]  
y = dataframe.iloc[:, -1:]

###  6. Data Normalization (Min-Max Scaling)

#### Normalizing the input and the output using:
$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

In [6]:
scaler = MinMaxScaler()  
X_scaled = scaler.fit_transform(X.values)  
new_X = pd.DataFrame(data=X_scaled, columns=X.columns)

### 7. Splitting into Train and Test Sets

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    new_X, y, test_size=0.2, random_state=1, shuffle=True, stratify=y
)

### 8. Converting Data to PyTorch Tensors

In [8]:
train_input_tensor = torch.from_numpy(X_train.values).float()  
train_label_tensor = torch.from_numpy(y_train['status'].values).float().unsqueeze(1)  
val_input_tensor = torch.from_numpy(X_test.values).float()  
val_label_tensor = torch.from_numpy(y_test['status'].values).float().unsqueeze(1)

### 9. Creating DataLoaders

In [9]:
train_dataset = TensorDataset(train_input_tensor, train_label_tensor)  
val_dataset = TensorDataset(val_input_tensor, val_label_tensor)  
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)  
val_loader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=True)

### 10. Defining the Neural Network (MLP)
The neural network begins with 87 input features and projects them into a higher-dimensional space using a fully connected layer `Linear(87, 300)`, followed by a ReLU activation: $ \text{ReLU}(x) = \max(0, x) $ to introduce non-linearity. Next, batch normalization is applied: for each feature $ x_i $, the output is $ \hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} $, where $ \mu $ and $ \sigma^2 $ are the mini-batch mean and variance, and $ \epsilon $ is a small constant to avoid division by zero. The normalized value $ \hat{x}_i $ is then scaled and shifted using learnable parameters $ \gamma $ and $ \beta $: $ y_i = \gamma \hat{x}_i + \beta $. This helps stabilize and accelerate training. A dropout layer with $ p = 0.4 $ follows, randomly disabling 40\% of neurons during training to prevent overfitting. The output then passes through another fully connected layer `Linear(300, 100)`, again followed by ReLU, batch normalization, and no dropout this time. Finally, a `Linear(100, 1)` layer reduces the representation to a single output, and a sigmoid activation $ \sigma(x) = \frac{1}{1 + e^{-x}} $ converts it to a probability. The network is trained using binary cross-entropy loss: $ \mathcal{L} = -[y \log(p) + (1 - y)\log(1 - p)] $, where $ y $ is the true label and $ p $ is the predicted probability.


<p align="center">
  <img src="Neural%20network.png" width="900"/>
</p>

In [10]:
class MLP(nn.Module):
    def __init__(self, dropout=0.4):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=87, out_features=300),
            nn.ReLU(),
            nn.BatchNorm1d(num_features=300),
            nn.Dropout(p=dropout),
            nn.Linear(in_features=300, out_features=100),
            nn.ReLU(),
            nn.BatchNorm1d(num_features=100),
            nn.Linear(in_features=100, out_features=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

### 11. Setting Up Training Components
Here's `Adam` optimizer which adjust the learning rate.
`BCE Loss` =  $ \mathcal{L} = -[y \log(p) + (1 - y)\log(1 - p)] $

In [11]:
model = MLP(dropout=0.4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)
criterion = nn.BCELoss()

### 12. Training the Model

In [12]:
epochs = 12
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/12], Loss: 0.2144
Epoch [2/12], Loss: 0.1712
Epoch [3/12], Loss: 0.1554
Epoch [4/12], Loss: 0.1488
Epoch [5/12], Loss: 0.1342
Epoch [6/12], Loss: 0.1391
Epoch [7/12], Loss: 0.1247
Epoch [8/12], Loss: 0.1319
Epoch [9/12], Loss: 0.1194
Epoch [10/12], Loss: 0.1123
Epoch [11/12], Loss: 0.1146
Epoch [12/12], Loss: 0.1104


### 13. Evaluating the Model
disable gradient computation using `torch.no_grad()` for evaluation.
Model predictions are thresholded at 0.5.<br>
Accuracy computed as $$Accuracy = \frac{Correct Predictions}{Total Predictions} \times 100$$

In [13]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        predicted = (outputs > 0.43).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

Validation Accuracy: 95.54%


#### Now let's try another Model 

The `AnotherMLP` model is a Multi-Layer Perceptron (MLP) with four hidden layers and dropout regularization. The input layer consists of 87 features, which are passed through successive hidden layers with 512, 256, 128, and 64 neurons, each followed by a Leaky ReLU activation $f(x) = \max(0.01 \cdot x, x)$ to avoid the dying ReLU problem, batch normalization $\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$ for stability, and a dropout rate of 0.4 to prevent overfitting. The output layer has one neuron with a Sigmoid activation $\sigma(x) = \frac{1}{1 + e^{-x}}$, producing a probability for binary classification. <br>The model is trained using Binary Cross-Entropy loss $\mathcal{L} = -[y \log(\hat{y}) + (1 - y)\log(1 - \hat{y})]$, which penalizes the difference between the predicted probability $\hat{y}$ and the actual label $y$. Batch normalization accelerates training by reducing internal covariate shift, and the Leaky ReLU activation ensures better gradient flow during training.
<p align="center">
  <img src="anothernn.png" width="900"/>
</p>


In [ ]:
class AnotherMLP(nn.Module):
    def __init__(self, dropout=0.4):
        super(AnotherMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=87, out_features=512),  
            nn.LeakyReLU(negative_slope=0.01), 
            nn.BatchNorm1d(num_features=512),
            nn.Dropout(p=dropout),
            
            nn.Linear(in_features=512, out_features=256),  
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm1d(num_features=256),
            nn.Dropout(p=dropout),
            
            nn.Linear(in_features=256, out_features=128),  
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm1d(num_features=128),
            nn.Dropout(p=dropout),
            
            nn.Linear(in_features=128, out_features=64),  
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm1d(num_features=64),
            nn.Dropout(p=dropout),
            
            nn.Linear(in_features=64, out_features=1),  
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

In [17]:
model_2 = AnotherMLP(dropout=0.4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_2 = model_2.to(device)

# Optimizer and Loss function
optimizer = torch.optim.Adam(params=model_2.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [18]:
epochs = 12
for epoch in range(epochs):
    model_2.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_2(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/12], Loss: 0.2549
Epoch [2/12], Loss: 0.1900
Epoch [3/12], Loss: 0.1722
Epoch [4/12], Loss: 0.1675
Epoch [5/12], Loss: 0.1602
Epoch [6/12], Loss: 0.1545
Epoch [7/12], Loss: 0.1526
Epoch [8/12], Loss: 0.1493
Epoch [9/12], Loss: 0.1439
Epoch [10/12], Loss: 0.1362
Epoch [11/12], Loss: 0.1360
Epoch [12/12], Loss: 0.1413


In [19]:
model_2.eval()  # Set the model to evaluation mode
correct = 0  # Initialize the count of correct predictions
total = 0  # Initialize the total number of samples

# No gradients required during evaluation
with torch.no_grad():
    for inputs, labels in val_loader:  # Iterate over batches in the validation loader
        inputs, labels = inputs.to(device), labels.to(device)  # Move inputs and labels to the device (GPU/CPU)
        outputs = model_2(inputs)  # Get model predictions
        predicted = (outputs > 0.43).float()  # Apply threshold of 0.43 to the model's output to get binary predictions (0 or 1)
        
        total += labels.size(0)  # Add the batch size to the total number of samples
        correct += (predicted == labels).sum().item()  # Count the correct predictions in the batch

# Calculate and print the validation accuracy
validation_accuracy = 100 * correct / total
print(f"Validation Accuracy: {validation_accuracy:.2f}%")

Validation Accuracy: 96.15%
